<a href="https://colab.research.google.com/github/EsarFatima/MachineLearning-flyrank-/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

My label (`is_declining_label`, from `trend_direction == "down"`) is a yes/no observed label, not a
ranking or grouping problem. Per the toolkit table, that shape starts with **Logistic Regression,
then Random Forest** — readable first, stronger second. I'm skipping clustering (not trying to name
groups of pages) and skipping Gradient Boosting this pass (30k rows, ~30 clients — more model than
the data needs; simplicity is a feature). I also run **permutation importance** on the Random Forest
since importances from a fit alone can be misleading.

In [ ]:
import os, sys, subprocess
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/EsarFatima/MachineLearning-flyrank-"
REPO_DIR = "MachineLearning-flyrank-"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

RANDOM_STATE = 42
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Label only -- never used as a feature below.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

print(df.shape[0], "rows |", df["client_id"].nunique(), "clients |",
      "base rate:", round(df["is_declining_label"].mean(), 3))

30000 rows | 32 clients | base rate: 0.542


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

This dataset repeats rows per **client** (32 clients, 30,000 pages). A random row split would let
the model memorize client-level habits and fake skill on pages from clients already seen elsewhere
in training. The honest question is "does this work on a client it has never seen?" — so I use a
**grouped split by `client_id`** (`GroupShuffleSplit`, 80/20, seed fixed at 42) and confirm zero
client overlap between train and test.

Features only use information knowable at scoring time — no `trend_pct` / `trend_direction` (the
label), no future window. Missing `search_volume`/`competition`/`cpc` and `word_count`/`char_count`
are real gaps in this data (per the data dictionary — `feedly article` rows have no keyword data,
some rows have no word count), so I keep explicit `has_keyword_data` / `has_word_count` flags
instead of silently filling with 0 and pretending that's a real value.

In [ ]:
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])

num_features = ["content_age_days", "days_since_last_update", "log_impressions_90d",
                "avg_position", "ctr", "engagement_rate", "search_volume", "competition",
                "word_count", "has_keyword_data", "has_word_count"]
cat_features = ["content_type", "main_intent"]

X_num = df[num_features].replace([np.inf, -np.inf], np.nan).fillna(0)
X_cat = pd.get_dummies(df[cat_features].fillna("unknown"), drop_first=True)
X = pd.concat([X_num, X_cat], axis=1)
y = df["is_declining_label"]
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx, test_idx = next(gss.split(X, y, groups))

Xtr, Xte = X.iloc[train_idx], X.iloc[test_idx]
ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

overlap = set(groups.iloc[train_idx]) & set(groups.iloc[test_idx])
print("train rows:", len(train_idx), "| test rows:", len(test_idx))
print("train clients:", groups.iloc[train_idx].nunique(), "| test clients:", groups.iloc[test_idx].nunique())
print("client overlap between train/test:", len(overlap), "(must be 0)")

train rows: 23837 | test rows: 6163
train clients: 25 | test clients: 7
client overlap between train/test: 0 (must be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Week-4 baseline rule, recomputed on THESE test rows only -- same split, same fairness.
stale = (df["days_since_last_update"] >= 90).astype(int)
visible = ((df["impressions_90d"] >= 300) & (df["impressions_90d"] < 30000)).astype(int)
baseline_score_te = (stale * visible * df["impressions_90d"]).iloc[test_idx]

# Logistic Regression -- scaled, class-weighted for the ~51/49 label balance
scaler = StandardScaler()
Xtr_s = scaler.fit_transform(Xtr)
Xte_s = scaler.transform(Xte)

logreg = LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE)
logreg.fit(Xtr_s, ytr)
logreg_scores_te = logreg.predict_proba(Xte_s)[:, 1]

# Random Forest -- shallow and leaf-floored on purpose (small client count, avoid overfitting)
rf = RandomForestClassifier(n_estimators=300, max_depth=6, min_samples_leaf=20,
                             class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(Xtr, ytr)
rf_scores_te = rf.predict_proba(Xte)[:, 1]

rows = []
for name, scores in [("baseline (w04 rule)", baseline_score_te),
                      ("logistic regression", logreg_scores_te),
                      ("random forest", rf_scores_te)]:
    rows.append([name, roc_auc_score(yte, scores),
                 precision_at_k(scores, yte, 10),
                 precision_at_k(scores, yte, 20),
                 precision_at_k(scores, yte, 50)])

comparison = pd.DataFrame(rows, columns=["model", "auc", "p@10", "p@20", "p@50"]).round(3)
print(comparison.to_string(index=False))
print("base rate (test split):", round(yte.mean(), 3), "| n =", len(yte))

print("\nRandom Forest wins on AUC (0.591 vs 0.492) and on P@20/P@50 -- both models beat the")
print("baseline there. But the baseline wins at P@10 (0.60 vs 0.30-0.40): with only 10 rows that's")
print("a small, noisy top-of-queue effect, not proof the rule is secretly better. Reporting it")
print("anyway -- a metric that flips at a different K IS the finding, not something to hide.")

              model   auc  p@10  p@20  p@50
baseline (w04 rule) 0.492   0.6  0.45  0.38
logistic regression 0.541   0.3  0.50  0.56
      random forest 0.591   0.4  0.50  0.56
base rate (test split): 0.511 | n = 6163

Random Forest wins on AUC (0.591 vs 0.492) and on P@20/P@50 -- both models beat the
baseline there. But the baseline wins at P@10 (0.60 vs 0.30-0.40): with only 10 rows that's
a small, noisy top-of-queue effect, not proof the rule is secretly better. Reporting it
anyway -- a metric that flips at a different K IS the finding, not something to hide.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("Top features (RF built-in importance):")
print(importances.head(6))

perm = permutation_importance(rf, Xte, yte, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1)
perm_importances = pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False)
print("\nTop features (permutation importance, shuffled on held-out test):")
print(perm_importances.head(6))

test_df = df.iloc[test_idx].copy()
test_df["rf_score"] = rf_scores_te
test_df["pred"] = (test_df["rf_score"] >= 0.5).astype(int)

false_positives = test_df[(test_df["pred"] == 1) & (test_df["is_declining_label"] == 0)]
false_negatives = test_df[(test_df["pred"] == 0) & (test_df["is_declining_label"] == 1)]

print(f"\n{len(false_positives)} false positives, {len(false_negatives)} false negatives, "
      f"out of {len(test_df)} test rows")

cols = ["content_id", "avg_position", "content_age_days", "impressions_90d", "engagement_rate"]
print("\n3 false positives (flagged as declining, actually stable/up):")
print(false_positives[cols].head(3).to_string(index=False))
print("\n3 false negatives (missed, actually declining):")
print(false_negatives[cols].head(3).to_string(index=False))

print("\n--- Interpretation ---")
print("Leans on: log_impressions_90d, avg_position, content_age_days -- a believable story (old,")
print("weakly-ranked pages with a recent traffic shift), not a suspicious one. No label sibling")
print("(trend_pct/trend_direction) is in the feature set, and no single feature is 'too good'.")
print("Errors: FP and FN counts are close (not skewed to one failure mode), and both groups cluster")
print("near engagement_rate = 0 -- a floor value in this data, not a real zero -- meaning the model")
print("has the least signal exactly where engagement data is thin.")

Top features (RF built-in importance):
log_impressions_90d       0.301089
avg_position              0.196744
content_age_days          0.170891
word_count                0.105550
ctr                       0.049129
days_since_last_update    0.041054
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.